# EONET API → SQLite  
## Notebook z Projektem bazy danych oraz importem bazy danych


In [2]:
from __future__ import annotations



import json
import sqlite3
from contextlib import closing
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import requests

In [3]:
# Konfiguracja podstawowa:
# - API_URL: endpoint, z którego będziemy czytać dane
# - DB_PATH: ścieżka do lokalnej bazy SQLite
# - TIMEOUT_SECONDS: maksymalny czas oczekiwania na odpowiedź API
# - LAUNCH_LIMIT: ile rekordów pobrać w jednym wywołaniu

API_URL_EVENTS = "https://eonet.gsfc.nasa.gov/api/v3/events"
API_URL_CATEGORIES = "https://eonet.gsfc.nasa.gov/api/v3/categories"
API_URL_SOURCES = "https://eonet.gsfc.nasa.gov/api/v3/sources"
TIMEOUT_SECONDS = 30
EVENT_LIMIT = 20

## Tworzenie schematu bazy danych

In [4]:
DB_PATH = "eonet.sqlite3"

schema = """
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS event_geometry;
DROP TABLE IF EXISTS event_source;
DROP TABLE IF EXISTS event_category;
DROP TABLE IF EXISTS geometry;
DROP TABLE IF EXISTS geometry_type;
DROP TABLE IF EXISTS event;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS source;
DROP TABLE IF EXISTS import_log;

CREATE TABLE event (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    api_id TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    description TEXT,
    "start" DATETIME,
    "end" DATETIME,
    state TEXT,
    county TEXT
);

CREATE TABLE category (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    api_id TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    description TEXT
);

CREATE TABLE source (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    abbreviation TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    link TEXT
);

CREATE TABLE geometry_type (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT NOT NULL UNIQUE
);

CREATE TABLE geometry (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    latitude REAL,
    longitude REAL,
    type_id INTEGER NOT NULL,
    magnitude REAL,
    unit TEXT,
    date DATETIME,
    FOREIGN KEY (type_id)
        REFERENCES geometry_type(id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

CREATE TABLE event_category (
    event_id INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    PRIMARY KEY (event_id, category_id),
    FOREIGN KEY (event_id)
        REFERENCES event(id)
        ON UPDATE CASCADE
        ON DELETE CASCADE,
    FOREIGN KEY (category_id)
        REFERENCES category(id)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);

CREATE TABLE event_source (
    event_id INTEGER NOT NULL,
    source_id INTEGER NOT NULL,
    PRIMARY KEY (event_id, source_id),
    FOREIGN KEY (event_id)
        REFERENCES event(id)
        ON UPDATE CASCADE
        ON DELETE CASCADE,
    FOREIGN KEY (source_id)
        REFERENCES source(id)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);

CREATE TABLE event_geometry (
    event_id INTEGER NOT NULL,
    geometry_id INTEGER NOT NULL,
    PRIMARY KEY (event_id, geometry_id),
    FOREIGN KEY (event_id)
        REFERENCES event(id)
        ON UPDATE CASCADE
        ON DELETE CASCADE,
    FOREIGN KEY (geometry_id)
        REFERENCES geometry(id)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);

CREATE TABLE import_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    imported_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    api_name TEXT NOT NULL,
    endpoint TEXT NOT NULL,
    records_received INTEGER NOT NULL DEFAULT 0
        CHECK (records_received >= 0),
    records_saved INTEGER NOT NULL DEFAULT 0
        CHECK (records_saved >= 0),
    status TEXT NOT NULL
        CHECK (status IN ('SUCCESS', 'ERROR')),
    error_message TEXT
);

CREATE INDEX idx_event_category_category_id
ON event_category(category_id);

CREATE INDEX idx_event_source_source_id
ON event_source(source_id);

CREATE INDEX idx_event_geometry_geometry_id
ON event_geometry(geometry_id);

CREATE INDEX idx_geometry_date
ON geometry(date);

INSERT INTO geometry_type(type) VALUES ('Point');
INSERT INTO geometry_type(type) VALUES ('Polygon');
INSERT INTO geometry_type(type) VALUES ('LineString');
"""

with closing(sqlite3.connect(DB_PATH)) as conn:
    conn.executescript(schema)
    conn.commit()

print("Schemat bazy danych został utworzony.")

Schemat bazy danych został utworzony.


In [5]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    counts_df = pd.read_sql_query(
        """
        SELECT 'event' AS table_name, COUNT(*) AS rows_count FROM event
        UNION ALL
        SELECT 'category', COUNT(*) FROM category
        UNION ALL
        SELECT 'source', COUNT(*) FROM source
        UNION ALL
        SELECT 'geometry', COUNT(*) FROM geometry
        UNION ALL
        SELECT 'geometry_type', COUNT(*) FROM geometry_type
        UNION ALL
        SELECT 'event_category', COUNT(*) FROM event_category
        UNION ALL
        SELECT 'event_source', COUNT(*) FROM event_source
        UNION ALL
        SELECT 'event_geometry', COUNT(*) FROM event_geometry
        UNION ALL
        SELECT 'import_log', COUNT(*) FROM import_log
        """,
        conn,
    )

display(counts_df)

,table_name,rows_count
0,event,0
1,category,0
2,source,0
3,geometry,0
4,geometry_type,3
5,event_category,0
6,event_source,0
7,event_geometry,0
8,import_log,0


### Skrypt tworzący baze

In [6]:

DROP TABLE IF EXISTS import_log;
DROP TABLE IF EXISTS event_category;
DROP TABLE IF EXISTS event_source;
DROP TABLE IF EXISTS event_geometry;
DROP TABLE IF EXISTS source;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS geometry;
DROP TABLE IF EXISTS geometry_type;
DROP TABLE IF EXISTS event;

CREATE TABLE event (
	id INTEGER PRIMARY KEY, -- Przychodzi z API jako EONET_x zapisujemy tylko x
	title  VARCHAR(255) NOT NULL, -- Do przecinka z tego co przydchodzi w polu title
	description TEXT,
	start_date TIMESTAMP NOT NULL,
	end_date TIMESTAMP,
	state VARCHAR(100), -- do zastanowienia czy tak czy osobna tabela
	county VARCHAR(100)
);

CREATE TABLE geometry_type (
	id SERIAL PRIMARY KEY,
	type VARCHAR(100) NOT NULL
);

CREATE TABLE geometry (
	id SERIAL PRIMARY KEY, -- Własny z sql
	latitude FLOAT NOT NULL CHECK (90 >= latitude AND latitude >= -90), --latitude i longitude są w takich zakresach
	longitude FLOAT NOT NULL CHECK (180 >= longitude AND longitude >= -180),
	type_id INTEGER NOT NULL,
	magnitude FLOAT,
	unit FLOAT,
	measure_date TIMESTAMP NOT NULL,

	CONSTRAINT fk_geometry_type_geometry
		FOREIGN KEY (type_id)
		REFERENCES geometry_type(id)
		ON UPDATE CASCADE
		ON DELETE RESTRICT --nie chcemy usunąć typu punktu który jest zapisany w bazie
);

CREATE TABLE category (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(10) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
);

CREATE TABLE source (
	id SERIAL PRIMARY KEY,
	title VARCHAR(255) NOT NULL,
	description TEXT
	
);

CREATE TABLE event_category (
	event_id INTEGER NOT NULL,
	category_id INTEGER NOT NULL,

	CONSTRAINT pk_event_category
		PRIMARY KEY (event_id, category_id),
		
	CONSTRAINT fk_event_category_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_category_category
		FOREIGN KEY (category_id)
		REFERENCES category(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_source (
	event_id INTEGER  NOT NULL,
	source_id INTEGER NOT  NULL,

	
	CONSTRAINT pk_event_source
		PRIMARY KEY (event_id, source_id),

	CONSTRAINT fk_event_source_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_source_source
		FOREIGN KEY (source_id)
		REFERENCES source(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_geometry (
	event_id INTEGER NOT NULL,
	geometry_id INTEGER NOT NULL,


	CONSTRAINT pk_event_geometry
		PRIMARY KEY (event_id, geometry_id),

	CONSTRAINT fk_event_geometry_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_geometry
		FOREIGN KEY (geometry_id)
		REFERENCES geometry(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE

);

CREATE TABLE import_log (
    id SERIAL PRIMARY KEY,
    imported_at TIMESTAMP NOT NULL,
    apiname VARCHAR(255) NOT NULL,
    endpoint VARCHAR(255) NOT NULL,
    records_received INTEGER NOT NULL,
    records_saved INTEGER NOT NULL,
    status VARCHAR(100) NOT NULL,
    error_message TEXT NOT NULL
)

SyntaxError: invalid syntax (1493467006.py, line 1)

## Pobieranie danych z API


In [ ]:
from typing import Any
from time import sleep

import pandas as pd
import requests

API_URL_EVENTS = "https://eonet.gsfc.nasa.gov/api/v3/events"

TIMEOUT_SECONDS = 30
EVENT_LIMIT = 100
DAYS_BACK = 90

USA_BBOX = "-124.85,49.38,-66.89,24.40"


def fetch_events_from_usa() -> list[dict[str, Any]]:
    params = {
        "bbox": USA_BBOX,
        "status": "all",
        "days": DAYS_BACK,
        "limit": EVENT_LIMIT,
    }

    for attempt in range(1, 4):
        try:
            response = requests.get(
                API_URL_EVENTS,
                params=params,
                headers={"Accept": "application/json"},
                timeout=TIMEOUT_SECONDS,
            )

            response.raise_for_status()

            payload = response.json()
            events = payload.get("events", [])

            if not isinstance(events, list):
                raise ValueError("API zwróciło dane w nieoczekiwanym formacie.")

            return events

        except requests.RequestException as error:
            print(f"Próba {attempt}/3 nie powiodła się: {error}")

            if attempt < 3:
                sleep(3)

    return []


events = fetch_events_from_usa()

print(f"Liczba pobranych zdarzeń z USA: {len(events)}")

events_df = pd.DataFrame(
    [
        {
            "api_id": event.get("id"),
            "title": event.get("title"),
            "description": event.get("description"),
            "closed": event.get("closed"),
            "categories_count": len(event.get("categories", [])),
            "sources_count": len(event.get("sources", [])),
            "geometries_count": len(event.get("geometry", [])),
        }
        for event in events
    ]
)

display(events_df)


Liczba pobranych zdarzeń z USA: 100


,api_id,title,description,closed,categories_count,sources_count,geometries_count
0,EONET_20387,"Tower Wildfire, Millard, Utah","31 Miles SE from Nephi, UT",None,1,1,1
1,EONET_20383,"FRONTAGE Wildfire, Baker, Oregon","1 Miles S from HUNTINGTON, OR, OR",2026-06-07T00:00:00Z,1,1,1
2,EONET_20388,"Raven Creek Wildfire, Weston, Wyoming","13 Miles SE from Moorcroft, WY",None,1,1,1
3,EONET_20389,"CHESTNUT Wildfire, Chelan, Washington","4 Miles S from Chelan, WA",None,1,1,1
4,EONET_20390,"Blue Ridge Wildfire, Owyhee, Idaho","16 Miles W from Buhl, ID",2026-06-06T00:00:00Z,1,1,1
...,...,...,...,...,...,...,...
95,EONET_20098,"Eagle Lake Fire Wildfire, Hancock, Iowa",None,None,1,1,1
96,EONET_20204,"DeSoto BB 8366 RX Prescribed Fire, Harrison, M...",None,None,1,1,1
97,EONET_20206,"Chickasawhay CPT 439 441 RX Prescribed Fire, W...",None,None,1,1,1
98,EONET_20156,"SEVEN CABINS Wildfire, Lincoln, New Mexico","13 Miles NW from Capitan, NM",None,1,1,1


## Szybki podgląd danych z API

## Zapisywanie danych do tabel

## Uruchomienie pełnego importu

## Weryfikacja zawartości bazy danych